In [1]:
import os
from typing import Any, Dict, List, TypedDict
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime

from dotenv import load_dotenv
load_dotenv()

model = ChatOpenAI(model="gpt-4.1-mini")

In [2]:
# Real-world Example: Customer Support System with Context
# This demonstrates how ToolRuntime provides context to tools

USER_DATABASE = {
    "user123": {
        "name": "Alice Johnson", 
        "account_type": "Premium",
        "balance": 5000,
        "email": "alice@example.com",
        "support_tier": "Priority"
    },
    "user456": {
        "name": "Bob Smith",
        "account_type": "Standard", 
        "balance": 1200,
        "email": "bob@example.com",
        "support_tier": "Standard"
    }
}

class UserContext(TypedDict):
    user_id: str
    session_id: str

@tool
def get_account_info(runtime: ToolRuntime[UserContext]) -> str:
    """Get the current user's account information."""
    user_id = runtime.context["user_id"]
    
    if user_id in USER_DATABASE:
        user = USER_DATABASE[user_id]
        return f"""Account Information:
        Name: {user['name']}
        Type: {user['account_type']}  
        Balance: ${user['balance']}
        Email: {user['email']}
        Support Tier: {user['support_tier']}"""
    return "User not found in system"

@tool
def update_support_tier(new_tier: str, runtime: ToolRuntime[UserContext]) -> str:
    """Update user's support tier (Priority, Standard, Basic)."""
    user_id = runtime.context["user_id"]
    
    if user_id in USER_DATABASE:
        USER_DATABASE[user_id]['support_tier'] = new_tier
        return f"Support tier updated to {new_tier} for user {USER_DATABASE[user_id]['name']}"
    return "User not found"

# Create agent with context
agent = create_agent(
    model,
    tools=[get_account_info, update_support_tier],
    context_schema=UserContext,
    system_prompt="You are a customer support assistant. Help users with their account inquiries."
)

# Test with real agent - Context is automatically injected into tools
print("=== Testing Context Access ===")
result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's my current account information?"}]},
    context=UserContext(user_id="user123", session_id="session_001")
)

=== Testing Context Access ===


In [3]:
print("\nAgent Response:")
print(result["messages"][-1].content)


Agent Response:
Your current account information is as follows:
- Name: Alice Johnson
- Account Type: Premium
- Balance: $5000
- Email: alice@example.com
- Support Tier: Priority

If you need any further assistance or want to update any details, please let me know!


In [4]:
import os
from typing import List, TypedDict

from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore

from dotenv import load_dotenv
load_dotenv()

model = ChatOpenAI(model="gpt-4.1-mini")


class UserContext(TypedDict):
    user_id: str


def memory_namespace(user_id: str) -> tuple[str, ...]:
    return ("app", "memory", user_id)

In [5]:
LTM_KEY = "messages"


@tool
def save_ltm(data: str, runtime: ToolRuntime[UserContext]) -> str:
    """Save a line of information to this user's long-term memory."""
    store = runtime.store
    ns = memory_namespace(runtime.context["user_id"])

    item = store.get(ns, LTM_KEY)
    if item is None:
        messages: List[str] = []
    else:
        messages = list(item.value) if isinstance(item.value, list) else [str(item.value)]

    messages.append(data)
    store.put(ns, LTM_KEY, messages)
    print(f"Saving to LTM for {runtime.context['user_id']}:", data)
    return "Data saved to long-term memory."


@tool
def retrieve_ltm(data: str, runtime: ToolRuntime[UserContext]) -> str:
    """Retrieve this user's long-term memory (data arg can hint what to look for)."""
    store = runtime.store
    ns = memory_namespace(runtime.context["user_id"])

    print(f"Retrieving from LTM for {runtime.context['user_id']}")
    item = store.get(ns, LTM_KEY)
    if item is None:
        return "No data in long-term memory."
    return str(item.value)

In [6]:
agent = create_agent(
    tools=[save_ltm, retrieve_ltm],
    model=model,
    store=InMemoryStore(),
    checkpointer=InMemorySaver(),
    context_schema=UserContext,
    system_prompt="""You are a helpful assistant that can answer any queries.
If the user shares personal information, preferences, or explicit approvals,
save that information using the save_ltm tool.
Before answering a query when prior facts might matter, retrieve long-term
memory using retrieve_ltm and use that context in your answer.
""",
)

In [7]:
config_vg_t1 = {"configurable": {"thread_id": "vg-chat-1"}}
vg = UserContext(user_id="vg")

response_1 = agent.invoke(
    {"messages": [{"role": "user", "content": "My name is Vaibhav"}]},
    config=config_vg_t1,
    context=vg,
)
response_1

Saving to LTM for vg: User's name is Vaibhav


{'messages': [HumanMessage(content='My name is Vaibhav', additional_kwargs={}, response_metadata={}, id='8354bf55-07d0-411b-a7e6-0ba5fc0de1d5'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 148, 'total_tokens': 169, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_81b7a13404', 'id': 'chatcmpl-ETOjuWhoJH2CqWVem7c8ll4QICGpF', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0ec90-f6b5-77e0-9fa3-7b764f0dd407-0', tool_calls=[{'name': 'save_ltm', 'args': {'data': "User's name is Vaibhav"}, 'id': 'call_zxYs8Itg9kHMbMv42u

In [8]:
config_vg_t2 = {"configurable": {"thread_id": "vg-chat-2"}}

response_2 = agent.invoke(
    {"messages": [{"role": "user", "content": "What is my name?"}]},
    config=config_vg_t2,
    context=vg,
)
response_2

Retrieving from LTM for vg


{'messages': [HumanMessage(content='What is my name?', additional_kwargs={}, response_metadata={}, id='eb85c5f5-4c41-4a86-8c53-710412df83e1'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 147, 'total_tokens': 162, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_81b7a13404', 'id': 'chatcmpl-ETOkTj5gEbyF40EICNAniLd1N1eJ5', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0ec91-7da4-7e20-998c-d420c735e6d3-0', tool_calls=[{'name': 'retrieve_ltm', 'args': {'data': 'name'}, 'id': 'call_m7ZFy345nKpiwaXygfl1RyBe', 'type':

In [9]:
config_alex_t1 = {"configurable": {"thread_id": "alex-chat-1"}}
alex = UserContext(user_id="alex")

response_3 = agent.invoke(
    {"messages": [{"role": "user", "content": "What is my name?"}]},
    config=config_alex_t1,
    context=alex,
)
response_3

Retrieving from LTM for alex


{'messages': [HumanMessage(content='What is my name?', additional_kwargs={}, response_metadata={}, id='37be6d1f-dbfe-4653-95fa-6acb012e549a'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 147, 'total_tokens': 162, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_81b7a13404', 'id': 'chatcmpl-ETOksN9SKASmMbkawPiKDjY4uJDpf', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0ec91-dfdd-7080-89c7-9af26766bdd0-0', tool_calls=[{'name': 'retrieve_ltm', 'args': {'data': 'name'}, 'id': 'call_W4yQmWhql6jcCq7X1eyDtcjM', 'type':